In [ ]:
import pandas as pd

data = pd.read_json("datasets/alpaca_data_cleaned-dutch.jsonl", lines=True)

def merge_instruction_into_input(df, instruction_col="instruction", input_col="input"):
    df = df.copy()
    df["prompt"] = (
        df[instruction_col].fillna("").str.strip()
        + "\ninput: "
        + df[input_col].fillna("").str.strip()
    ).str.strip()
    return df

# instructions and input in one "prompt" column (keeps original columns intact)
data = merge_instruction_into_input(data)

In [5]:
def split_train_test(data, train_frac=0.8, random_state=42, reset_idx=True):
    train_data = data.sample(frac=train_frac, random_state=random_state)
    test_data = data.drop(train_data.index)

    if reset_idx:
        train_data = train_data.reset_index(drop=True)
        test_data = test_data.reset_index(drop=True)

    return train_data, test_data


train_data, test_data = split_train_test(data, train_frac=0.8, random_state=42)

print(f"Train size: {len(train_data)}")
print(f"Test size: {len(test_data)}")

# save train and test set - ensure consistency in sampling
train_data.to_json("datasets/alpaca_train.jsonl", orient="records", lines=True)
test_data.to_json("datasets/alpaca_test.jsonl", orient="records", lines=True)

Train size: 41370
Test size: 10342


In [ ]:
import json
import sys
from pathlib import Path

# Ensure project root is on sys.path so `finetuning` package is importable.
_PROJECT_ROOT = str(Path(".").resolve())
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

from finetuning.prompts import build_training_text
from transformers import AutoTokenizer

with open("configs/qlora_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

model_name = config["model"]["name"]
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

from datasets import Dataset

hf_dataset = Dataset.from_pandas(train_data)

def formatting_prompts_func(batch):
    inputs = batch["prompt"]
    outputs = batch["output"]
    texts = []
    for user_input, assistant_output in zip(inputs, outputs):
        text = build_training_text(tokenizer, user_input, assistant_output)
        texts.append(text)
    return {"text": texts}

formatted_train_data = hf_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=hf_dataset.column_names,
)

# Save formatted dataset to disk
output_path = "datasets/alpaca_train_formatted"
formatted_train_data.save_to_disk(output_path)
print(f"Saved {len(formatted_train_data)} examples to {output_path}")
print(f"Sample:\n{formatted_train_data[0]['text'][:300]}...")

Saving the dataset (1/1 shards): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 41370/41370 [00:00<00:00, 70484.74 examples/s]


Saved 41370 examples to datasets/alpaca_train_formatted
Sample:
Beantwoord de volgende vraag zo goed mogelijk.

### Vraag:
Schrijf een gedicht van 7 regels met behulp van de gegeven woorden
input: paraplu, rail, water, lucht, gras, blad

### Antwoord:
Onder de don...


In [3]:
from datasets import load_from_disk

alpaca_train_formatted = load_from_disk("datasets/alpaca_train_formatted", keep_in_memory=True)
print(alpaca_train_formatted)
print(alpaca_train_formatted[0]["text"][:200] + "...")

Dataset({
    features: ['text'],
    num_rows: 41370
})
Beantwoord de volgende vraag zo goed mogelijk.

### Vraag:
Schrijf een gedicht van 7 regels met behulp van de gegeven woorden
input: paraplu, rail, water, lucht, gras, blad

### Antwoord:
Onder de don...


In [5]:
data

,id,instruction,input,output
0,7679,Schrijf een gedicht van 7 regels met behulp va...,Schrijf een gedicht van 7 regels met behulp va...,"Onder de donker wordende lucht,\nHet gras groe..."
1,30444,Beschrijf de voordelen van het volgende produc...,Beschrijf de voordelen van het volgende produc...,"Bluetooth koptelefoons bieden handsfree gemak,..."
2,21702,Herschrijf de zin hieronder,Herschrijf de zin hieronder\ninput: Een groot ...,Veel mensen
3,6133,Schrijf een algoritme dat klinkers in een gege...,Schrijf een algoritme dat klinkers in een gege...,Het algoritme kan als volgt worden geschreven:...
4,3228,Redigeer deze drie zinnen.,Redigeer deze drie zinnen.\ninput: Mijn naam i...,Mijn naam is ___________ en ik woon in _______...
...,...,...,...,...
41365,31264,"Op basis van de invoertekst, schrijf een overt...","Op basis van de invoertekst, schrijf een overt...",Schoolboeken moeten vrij beschikbaar zijn voor...
41366,43589,Schrijf tien vragen over het onderwerp interne...,Schrijf tien vragen over het onderwerp interne...,1. Wat is de belangrijkste stap bij het handha...
41367,38218,Genereer een op feiten gebaseerde verklaring d...,Genereer een op feiten gebaseerde verklaring d...,Het consumeren van chocolade kan een gevoel va...
41368,19466,Identificeer de belangrijkste onderwerpen die ...,Identificeer de belangrijkste onderwerpen die ...,De belangrijkste onderwerpen die besproken wor...
